# Parameter estimation for translational protein sector
This notebook was used to estimate the parameters for the translational protein sector as used in the revised proteome allocation model.

**Author: Samira L. van den Bogaard**

### Setting up the notebook environment

In [1]:
#package
import os
import pandas as pd
import numpy as np

if os.path.split(os.getcwd())[1] == 'Protein_sectors':
    os.chdir('../..')

from Scripts.pam_generation import setup_ecoli_pam
from Scripts.Protein_sectors.Sector_config_functions import (perform_linear_regression,
                                                                        reset_translational_sector,
                                                                        plot_translational_protein_vs_mu,
                                                                        get_model_simulations)

Loading PAModelpy modules version 0.0.3.3
Set parameter Username
Academic license - for non-commercial use only - expires 2025-03-06


In [2]:
#plotly in jupyter notebooks
from plotly.offline import plot, iplot, init_notebook_mode
import plotly.graph_objs as go
init_notebook_mode(connected=True)

In [3]:
#constants
#files and file paths
DATA_FILE_PATH = os.path.join('Data', 'Ecoli_phenotypes', 'Proteome','Schmidt-quantitative-proteomics_2010_parsed.xls')
PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_py.xls')
VALID_DATA_FILE_PATH = os.path.join('Data', 'Ecoli_phenotypes', 'Ecoli_phenotypes_py.xls')

#data parsing
NOT_IN_ANALYSIS_LIST = ['50 mM NaCl', '42°C', 'pH 6']#, 'glycerol + AA','LB']
PROTEIN_FRACTION = 0.258 #[g_prot/g_cdw]
MEASURED_PROTEIN_FRACTION = 0.55*0.55#0.55 * 0.55 #measured proteome vs g_p/gcdw in Ecoli
GLC_UPTAKE_RXN_ID = 'EX_glc__D_e'
BIOMASS_RXN_ID = 'BIOMASS_Ec_iML1515_core_75p37M'
OXYGEN_GRADIENT = [5,10,15,17,20]
MW_GLC = 180.15588 #g/mol


### Configure parameters and data
To establish the relation between translational protein and substrate uptake rate we use the well-established relation between translational protein and growth rate from the data from Schmidt et al. (2016). For that we set up the model with the sector constraints by [Alter et al. (2021)](https://journals.asm.org/doi/10.1128/mSystems.00625-20) and let it run without total protein constraint. This way we obtain a relationship between the translational protein and substrate uptake rate: the translational protein sector is determined by the relation to growth as published by [Schmidt et al. (2016)](https://www.nature.com/articles/nbt.3418) and the substrate uptake rate corresponding to the specific growth rate are predicted by the model. As the relationship between growth and translational protein seems to cease when the *E. coli* cells are entering overflow metabolism, we do not use the datapoints in the dataset from [Schmidt et al. (2016)](https://www.nature.com/articles/nbt.3418) which belong to growth rates in the overflow regime.

In [4]:
#set up model without total protein constraint to just model the linear relation between
pamodel = setup_ecoli_pam(total_protein=False, active_enzymes = False, unused_enzymes = False,
                          pam_data_file_path = os.path.join('Data', 
                                                            'proteinAllocationModel_iML1515_EnzymaticData_py.xls'),
                          sensitivity = False) # sensitivity disabled to speed things up
#get parameters
translational_sector = pamodel.sectors.get_by_id('TranslationalProteinSector')
tps_molmass = translational_sector.mol_mass[0]

Setting up the proteome allocation model iML1515

Total condition-dependent protein constraint is not added 

Add the following protein sector: TranslationalProteinSector

Done with setting up the proteome allocation model iML1515



In [5]:
#get the validation data
df_schmidt = pd.read_excel(DATA_FILE_PATH)
#removing conditions which we do not want to include in the analysis
for condition in NOT_IN_ANALYSIS_LIST:
    df_schmidt = df_schmidt[df_schmidt['Medium'] != condition]
    
#filter the range of growth rates for those who are nor influenced by more complex metabolic processes, such as:
#stationary growth
df_schmidt_nozero = df_schmidt[df_schmidt['Experimental Growth rate']>0.1]
#overflow metabolism
df_schmidt_nozero = df_schmidt_nozero[(df_schmidt_nozero['Experimental Growth rate']<0.4) | (df_schmidt_nozero['Experimental Growth rate']>0.7)]

The slope and intercept are derived from proteome measurements. We know these measurement are related to a total protein concentration of 0.32 g/g_cdw. This is why we need to correct the slope and intercept when it it incorporated into the model.

In [6]:
#get the relation with growth rate
slope, intercept = perform_linear_regression(x=df_schmidt_nozero['Experimental Growth rate'], y=df_schmidt_nozero['Translation, ribosomal structure and biogenesis']/100*MEASURED_PROTEIN_FRACTION/PROTEIN_FRACTION)

print('Translational sector vs growth rate using data from Schmidt et al (2016): \n Intercept: ',intercept*MEASURED_PROTEIN_FRACTION, 'g/g_cdw')
print('Slope: ',slope*MEASURED_PROTEIN_FRACTION, 'g/g_cdw/h')

Translational sector vs growth rate using data from Schmidt et al (2016): 
 Intercept:  0.04738115630907698 g/g_cdw
Slope:  0.04806975534478209 g/g_cdw/h


### Use the mu vs translational protein relationship to perform simulations with the PAM model
The predicted glucose uptake rates can be used to derive the relationship between glucose uptake and translational protein.

In [7]:
#adjust relation in model
trans_sector = pamodel.sectors.get_by_id('TranslationalProteinSector')

#adjust the translational protein sector
pamodel.change_sector_parameters(trans_sector, slope*MEASURED_PROTEIN_FRACTION,
                                 intercept*MEASURED_PROTEIN_FRACTION, lin_rxn_id = BIOMASS_RXN_ID)

Changing the slope and intercept of the TranslationalProteinSector
Changing slope from 36.8 to 48.06975534478209 mg/gcdw/h
Changing intercept from 49.92 to 47.38115630907698 mg/gcdw


In [8]:
#run the model
substrate_range = np.arange(-11,1,1)

#we shouldn't correct for measured total protein here, as we are comparing it to the measured data
simulation_results_bms = get_model_simulations(pamodel, GLC_UPTAKE_RXN_ID, BIOMASS_RXN_ID,
                                           substrate_range, intercept,slope) 


Running simulations with  -11 mmol/g_cdw/h of substrate going into the system
Running simulations with  -10 mmol/g_cdw/h of substrate going into the system
Running simulations with  -9 mmol/g_cdw/h of substrate going into the system
Running simulations with  -8 mmol/g_cdw/h of substrate going into the system
Running simulations with  -7 mmol/g_cdw/h of substrate going into the system
Running simulations with  -6 mmol/g_cdw/h of substrate going into the system
Running simulations with  -5 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -1 mmol/g_cdw/h of substrate going into the system
Running simulations with  0 mmol/g_cdw/h of substrate going into the system


In [9]:
#derive the relation to glucose uptake rate
#results are in g_transprot/g_prot and need to be converted to g/gcdw (glucose uptake should stay in mmol for compatibility with model)
slope_glc, intercept_glc = perform_linear_regression(
    x=simulation_results_bms[GLC_UPTAKE_RXN_ID], y=simulation_results_bms['translational_protein'])

print('Translational sector vs glucose uptake using data from Schmidt et al (2016): \n Intercept: ',intercept_glc *MEASURED_PROTEIN_FRACTION, ' g/g_cdw')
print('Slope: ',slope_glc*MEASURED_PROTEIN_FRACTION, ' g/mmol_glucose/h')


Translational sector vs glucose uptake using data from Schmidt et al (2016): 
 Intercept:  0.046136644909661115  g/g_cdw
Slope:  -0.004340256958025938  g/mmol_glucose/h


In [10]:
#visualization of results
plot_translational_protein_vs_mu(df_schmidt, simulation_results_bms, intercept_glc,
                                 slope_glc, 
                                 protein_fraction = PROTEIN_FRACTION, 
                                 measured_protein_fraction=MEASURED_PROTEIN_FRACTION,
                                 oxygen_results = None)

### Use the glucose vs translational protein relationship to perform simulations with the PAM model with only the translational protein sector

The result will be compared to the experimental data. The slope and intercept are derived from proteome measurements. We know these measurement are related to a total protein concentration of 0.32 g/g_cdw. This is why we need to correct the slope and intercept when it it incorporated into the model. In the comparison, this correction is not required.

In [11]:
#make a model with only translational protein sector
pamodel_transl = setup_ecoli_pam(total_protein=False ,active_enzymes=False, unused_enzymes=False,
                                 pam_data_file_path = os.path.join('Data', 
                                                            'proteinAllocationModel_iML1515_EnzymaticData_py.xls'),
                          sensitivity = False) # sensitivity disabled to speed things up

#adjust the translational protein sector
pamodel_transl = reset_translational_sector(pamodel_transl, slope_glc*MEASURED_PROTEIN_FRACTION, 
                                            intercept_glc*MEASURED_PROTEIN_FRACTION, new_id = GLC_UPTAKE_RXN_ID)

Setting up the proteome allocation model iML1515

Total condition-dependent protein constraint is not added 

Add the following protein sector: TranslationalProteinSector

Done with setting up the proteome allocation model iML1515

Removing the following protein sector: TranslationalProteinSector

Add the following protein sector: TranslationalProteinSector



In [12]:
simulation_results_glc = get_model_simulations(pamodel_transl, BIOMASS_RXN_ID, GLC_UPTAKE_RXN_ID,
                                           substrate_range, intercept_glc, slope_glc)

Running simulations with  -11 mmol/g_cdw/h of substrate going into the system
Running simulations with  -10 mmol/g_cdw/h of substrate going into the system
Running simulations with  -9 mmol/g_cdw/h of substrate going into the system
Running simulations with  -8 mmol/g_cdw/h of substrate going into the system
Running simulations with  -7 mmol/g_cdw/h of substrate going into the system
Running simulations with  -6 mmol/g_cdw/h of substrate going into the system
Running simulations with  -5 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -1 mmol/g_cdw/h of substrate going into the system
Running simulations with  0 mmol/g_cdw/h of substrate going into the system


In [13]:
#visualize results
plot_translational_protein_vs_mu(df_schmidt, simulation_results_glc, intercept_glc, slope_glc, 
                                 protein_fraction = PROTEIN_FRACTION, 
                                 measured_protein_fraction=MEASURED_PROTEIN_FRACTION,
                                 oxygen_results = None)

### Full PAM simulations (all sectors)
Now let's see if we can improve the fit through the datapoints from [Schmidt et al. (2016)](https://www.nature.com/articles/nbt.3418) if we use the newly obtained relationship in a PAM with all protein sectors.

In [14]:
#make a model with only translational protein sector
pamodel_full = setup_ecoli_pam(unused_enzymes =False, pam_data_file_path = os.path.join('Data', 
                                                            'proteinAllocationModel_iML1515_EnzymaticData_py.xls'),
                              sensitivity = False) # sensitivity disabled to speed things up

trans_sector = pamodel_full.sectors.get_by_id('TranslationalProteinSector')

#adjust the translational protein sector
pamodel_full.change_sector_parameters(trans_sector, slope_glc*MEASURED_PROTEIN_FRACTION, intercept_glc*MEASURED_PROTEIN_FRACTION, lin_rxn_id = GLC_UPTAKE_RXN_ID)

Setting up the proteome allocation model iML1515

Add total condition-dependent protein constraint
	Total protein concentration: 0.258 g/gDW

Add active protein sector

Add the following protein sector: TranslationalProteinSector

Done with setting up the proteome allocation model iML1515

Changing the slope and intercept of the TranslationalProteinSector
Changing slope from 36.8 to -4.340256958025938 mg/gcdw/h
Changing intercept from 49.92 to 46.136644909661115 mg/gcdw


In [15]:
print(trans_sector.slope, trans_sector.intercept)

-4.340256958025938 46.136644909661115


In [16]:
simulation_results = get_model_simulations(pamodel_full,BIOMASS_RXN_ID, GLC_UPTAKE_RXN_ID, 
                                           substrate_range, intercept_glc, slope_glc)

# # Also add the linear relation to growth rate to compare with the linear relation with glucose uptake
# slope, intercept = perform_linear_regression(simulation_results[BIOMASS_RXN_ID],
#                                                                 simulation_results['translational_protein'])

# #add extrapolation of results for complex growth medium (1.9 h-1)
# row = simulation_results.iloc[-1]
# row[BIOMASS_RXN_ID] = 1.9
# row['translational_protein'] = (intercept + slope*1.9)
# simulation_results.loc[len(simulation_results.index)] = row.to_list()

Running simulations with  -11 mmol/g_cdw/h of substrate going into the system
Running simulations with  -10 mmol/g_cdw/h of substrate going into the system
Running simulations with  -9 mmol/g_cdw/h of substrate going into the system
Running simulations with  -8 mmol/g_cdw/h of substrate going into the system
Running simulations with  -7 mmol/g_cdw/h of substrate going into the system
Running simulations with  -6 mmol/g_cdw/h of substrate going into the system
Running simulations with  -5 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -1 mmol/g_cdw/h of substrate going into the system
Running simulations with  0 mmol/g_cdw/h of substrate going into the system


In [17]:
#visualize results
plot_translational_protein_vs_mu(df_schmidt, simulation_results, intercept_glc, slope_glc, 
                                 protein_fraction = PROTEIN_FRACTION, 
                                 measured_protein_fraction=MEASURED_PROTEIN_FRACTION,
                                 oxygen_results = None)

### Oxygen limitation simulations
By limiting the amount of oxygen, the organism will switch to a fermentative phenotype. As a consequence, more protein space for translational protein is freed. This mimmics the effect of overflow metabolism on the amount of translational protein (this can be seen in the data). 

In [18]:
#run the simulations for different oxygen availabilities
o2_simulations = pd.DataFrame(columns = list(simulation_results.columns) + ['o2'])
for o2_level in OXYGEN_GRADIENT:
    print(f'\nRun model with {o2_level} mmol/g_cdw/h of oxygen going into the system')
    pamodel_transl.change_reaction_bounds('EX_o2_e', lower_bound = -o2_level, upper_bound = o2_level)
    simulation_results_o2 = get_model_simulations(pamodel_transl, GLC_UPTAKE_RXN_ID, BIOMASS_RXN_ID,
                                           substrate_range, intercept, slope)
    simulation_results_o2['o2'] = o2_level
    
    o2_simulations = pd.concat([o2_simulations, simulation_results_o2])


Run model with 5 mmol/g_cdw/h of oxygen going into the system
Running simulations with  -11 mmol/g_cdw/h of substrate going into the system
Running simulations with  -10 mmol/g_cdw/h of substrate going into the system
Running simulations with  -9 mmol/g_cdw/h of substrate going into the system
Running simulations with  -8 mmol/g_cdw/h of substrate going into the system
Running simulations with  -7 mmol/g_cdw/h of substrate going into the system
Running simulations with  -6 mmol/g_cdw/h of substrate going into the system
Running simulations with  -5 mmol/g_cdw/h of substrate going into the system
Running simulations with  -4 mmol/g_cdw/h of substrate going into the system
Running simulations with  -3 mmol/g_cdw/h of substrate going into the system
Running simulations with  -2 mmol/g_cdw/h of substrate going into the system
Running simulations with  -1 mmol/g_cdw/h of substrate going into the system
Running simulations with  0 mmol/g_cdw/h of substrate going into the system

Run model w

### Plot the results

In [19]:
# Also add the linear relation to growth rate to compare with the linear relation with glucose uptake
slope, intercept = perform_linear_regression(simulation_results[BIOMASS_RXN_ID],
                                                                simulation_results['translational_protein'])

#add extrapolation of results for complex growth medium (1.9 h-1)
row = o2_simulations.iloc[-1]
row[BIOMASS_RXN_ID] = 1.9
row['translational_protein'] = (intercept + slope*1.9)
o2_simulations.loc[len(o2_simulations.index)] = row.to_list()

In [20]:
#plotting vs mu
plot_translational_protein_vs_mu(df_schmidt, simulation_results, intercept_glc, slope_glc, 
                                 protein_fraction = PROTEIN_FRACTION, 
                                 measured_protein_fraction=MEASURED_PROTEIN_FRACTION, 
                                 oxygen_results = o2_simulations, oxygen_rxn_id = 'o2')

In [21]:
#plotting vs glucose uptake
fig = go.Figure()
fig.add_trace(go.Scatter(x=results[GLC_UPTAKE_RXN_ID], y=results['R_TranslationalProteinSector']*tps_molmass*1e-6/(1000),
                         mode='lines', line = dict(width = 5), name='no limitation'))
fig.update_xaxes(title_text = 'glucose uptake rate (mmol_glc/g_cdw/h)')
fig.update_yaxes(title_text = 'Translational protein sector (g_translationalprotein/g_protein)')

for result in results_o2:
    oxygen_concentration = result.iloc[-1, result.columns.get_loc(simulator.OXYGEN_UPTAKE_RXNID)]
    # translational unit is in mmol/g_cdw converting it to g/g_protein: * molmass*1e-6(scalingfactor)*100%/(1000(mg per g))
    fig.add_trace(go.Scatter(x=result[GLC_UPTAKE_RXN_ID],
                             y=result['R_TranslationalProteinSector']/ (1000),
                             mode='lines', line = dict(width = 5),
                             name = str(round(oxygen_concentration*100)/100)+' mmol_O2/g_cdw/h'))
fig.update_traces(marker_size=10)
fig.update_layout(font = dict(size = 14), title = 'Glucose limitation')
fig.add_annotation(x=2.2, y=0.45, showarrow=False,
            text=f"Intercept: {round(intercept_glc*1000)/1000} g_tprot/g_substrate")
fig.add_annotation(x=2.1, y=0.42,showarrow=False,
                   text=f"Slope: {round(slope_glc*1000)/1000} g_tprot/g_substrate/h")
fig.show()

NameError: name 'results' is not defined